In [1]:
import cv2
import numpy as np
from ultralytics import YOLO

# ─── CONFIGURATION ──────────────────────────────────────────────────────────

VIDEO_PATH = "./speed_test_2.mp4"
OUTPUT_PATH = "./speed_output1.mp4"
MODEL_PATH = "./detection_track.pt"

CONF_THRES = 0.25
IOU_THRES = 0.45
cap = cv2.VideoCapture(VIDEO_PATH)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()


In [2]:
height

1080

In [4]:

# 🔧 Adjust these based on your video
LINE_Y1 = int(height * 0.70)   # 70% down
LINE_Y2 = int(height * 0.85)   # 85% down
METERS_BETWEEN_LINES = 8.0

LINE_COLOR_1 = (255, 100, 0)
LINE_COLOR_2 = (0, 100, 255)
TEXT_COLOR = (0, 255, 80)

SPEED_PERSIST = 150   # keep speed visible longer

# ─── HELPERS ────────────────────────────────────────────────────────────────

def bottom_center(box):
    x1, y1, x2, y2 = box
    return int((x1 + x2) / 2), int(y2)

def crossed_line(prev_y, curr_y, line_y, tol=10):
    return (prev_y - line_y) * (curr_y - line_y) < 0 or abs(curr_y - line_y) < tol

# ─── STATE ──────────────────────────────────────────────────────────────────

prev_positions = {}
t1_timestamps = {}
speed_labels = {}

# ─── MAIN ───────────────────────────────────────────────────────────────────

def run():
    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"Frame size: {width}x{height}, FPS: {fps}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

    frame_idx = 0

    for result in model.track(
        source=VIDEO_PATH,
        conf=CONF_THRES,
        iou=IOU_THRES,
        tracker="bytetrack.yaml",
        persist=True,
        stream=True,
    ):
        frame = result.orig_img.copy()
        timestamp = frame_idx / fps

        # ── Draw lines ─────────────────────────────────────────
        cv2.line(frame, (0, LINE_Y1), (width, LINE_Y1), LINE_COLOR_1, 2)
        cv2.line(frame, (0, LINE_Y2), (width, LINE_Y2), LINE_COLOR_2, 2)

        cv2.putText(frame, "Line 1", (10, LINE_Y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_1, 2)
        cv2.putText(frame, "Line 2", (10, LINE_Y2 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, LINE_COLOR_2, 2)

        if result.boxes is not None and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.cpu().numpy().astype(int)

            for box, track_id in zip(boxes, ids):
                cx, cy = bottom_center(box)

                # 🔵 draw tracking point (DEBUG)
                cv2.circle(frame, (cx, cy), 4, (0, 255, 255), -1)

                if track_id in prev_positions:
                    _, prev_cy = prev_positions[track_id]

                    # Only consider downward motion
                    if cy > prev_cy:

                        if crossed_line(prev_cy, cy, LINE_Y1):
                            print(f"[DEBUG] ID {track_id} crossed LINE 1")
                            t1_timestamps[track_id] = timestamp

                        if crossed_line(prev_cy, cy, LINE_Y2):
                            print(f"[DEBUG] ID {track_id} crossed LINE 2")

                            if track_id in t1_timestamps:
                                dt = abs(timestamp - t1_timestamps.pop(track_id))

                                if dt > 0:
                                    speed = (METERS_BETWEEN_LINES / dt) * 3.6

                                    speed_labels[track_id] = {
                                        "speed_kmh": speed,
                                        "ttl": SPEED_PERSIST,
                                    }

                                    print(f"[SPEED] ID {track_id} → {speed:.2f} km/h")

                prev_positions[track_id] = (cx, cy)

                # ── Draw bounding box ─────────────────────
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 255), 2)
                cv2.putText(frame, f"ID:{track_id}", (x1, y1 - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 255), 1)

        # ── Draw speed labels ───────────────────────────────
        expired = []

        for tid, info in speed_labels.items():
            if tid in prev_positions:
                cx, cy = prev_positions[tid]

                speed = info['speed_kmh']
                label = f"{speed:.1f} km/h"

                y_text = max(30, cy - 20)

                # 🔴 If speed > 50 → RED + THICK
                if speed > 50:
                    color = (0, 0, 255)   # Red (BGR)
                    thickness = 4
                else:
                    color = TEXT_COLOR
                    thickness = 2

                # shadow
                cv2.putText(frame, label, (cx, y_text),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), thickness + 2)

                # main text
                cv2.putText(frame, label, (cx, y_text),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, thickness)

            info["ttl"] -= 1
            if info["ttl"] <= 0:
                expired.append(tid)

        for tid in expired:
            del speed_labels[tid]

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

    print(f"\n✅ Done! Output saved to: {OUTPUT_PATH}")


if __name__ == "__main__":
    run()

Frame size: 1920x1080, FPS: 50.0

video 1/1 (frame 1/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 7 cars, 24.0ms
video 1/1 (frame 2/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 7 cars, 25.8ms
video 1/1 (frame 3/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 21.1ms
video 1/1 (frame 4/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 6 cars, 21.0ms
video 1/1 (frame 5/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 29.1ms
video 1/1 (frame 6/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 6 cars, 1 scooter, 28.9ms
video 1/1 (frame 7/3000) c:\Users\Deep\Desktop\DAU_Projects\traffic\speed_estimation\speed_test_2.mp4: 384x640 6 cars, 25.1ms
video 1/1 (frame 8/3000) c:\Users\Deep\Desktop\DAU_